Note: Run this file to create spike count dataframe which is used to run ML Analyses

In [28]:
import pickle

# Third-party libraries
from LemonPy.utils_vmk import reverse_dictionary
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [4]:
# !pip install pyarrow==16.1.0

   ---------------------------------------- 0.0/25.9 MB ? eta -:--:--
   ------------ --------------------------- 8.1/25.9 MB 41.8 MB/s eta 0:00:01
   --------------------------- ------------ 17.8/25.9 MB 44.9 MB/s eta 0:00:01
   ---------------------------------------  25.4/25.9 MB 41.3 MB/s eta 0:00:01
   ---------------------------------------- 25.9/25.9 MB 37.2 MB/s  0:00:00


### Load spikes dataframe and visually responsive dictionary

In [ ]:
with open('visRes_units_dict.pickle', 'rb') as handle:
    visRes_dict = pickle.load(handle)

spikes_df = pd.read_parquet('operant_spikes.parquet')

In [40]:
spikes_df.head()

,cluster_id,spikes,trial,stim_id,trial_spikes,depth,cuid,group,set,region,et,cc,path,times
0.0,663,0.009700,0.0,0,0.009700,2580,CC067431_HP2_663,FX,1,v1,CC067431_HP2,CC067431,G:\Neuropixels\interval_operant_training\opera...,0.00
0.0,663,0.012133,0.0,0,0.012133,2580,CC067431_HP2_663,FX,1,v1,CC067431_HP2,CC067431,G:\Neuropixels\interval_operant_training\opera...,0.01
0.0,663,0.034767,0.0,0,0.034767,2580,CC067431_HP2_663,FX,1,v1,CC067431_HP2,CC067431,G:\Neuropixels\interval_operant_training\opera...,0.03
0.0,663,0.045933,0.0,0,0.045933,2580,CC067431_HP2_663,FX,1,v1,CC067431_HP2,CC067431,G:\Neuropixels\interval_operant_training\opera...,0.04
0.0,663,0.079700,0.0,0,0.079700,2580,CC067431_HP2_663,FX,1,v1,CC067431_HP2,CC067431,G:\Neuropixels\interval_operant_training\opera...,0.07


In [ ]:
spikes_df = spikes_df[spikes_df.region.isin(['hippo', 'v1'])]

spikes_df['group'] = spikes_df['group'].map({'A': 'WT', 'B': 'FX'})

times = np.round(np.arange(0, 3.01, 0.01), 3)
spikes_df['times'] = pd.cut(spikes_df['trial_spikes'], bins=times, labels=times[:-1])

spike_counts = spikes_df.groupby(['cuid', 'trial', 'times']).size().reset_index(name='spike_count')

result = pd.merge(spikes_df, spike_counts, on=['cuid', 'trial', 'times'], how='left')

spike_counts['et'] = spike_counts.cuid.str.slice(0, -4)

In [13]:
spike_counts.head()

,cuid,trial,times,spike_count,et
0,CC067431_HP2_185,0.0,0.00,0,CC067431_HP2
1,CC067431_HP2_185,0.0,0.01,0,CC067431_HP2
2,CC067431_HP2_185,0.0,0.02,0,CC067431_HP2
3,CC067431_HP2_185,0.0,0.03,0,CC067431_HP2
4,CC067431_HP2_185,0.0,0.04,0,CC067431_HP2


### Add additional info

In [ ]:
stim_order = np.array([0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0])

trials_number = 150

spike_counts['stim'] = spike_counts.trial.map(dict(zip(np.arange(trials_number),([int(i) for i in stim_order]))))

In [ ]:
map_et = spikes_df.groupby(['et']).group.unique().map(lambda x: x[0])
map_et = map_et.to_dict()

spike_counts['strain'] = spike_counts.et.map(map_et)

rev_visRes_dict = reverse_dictionary(visRes_dict)

spike_counts['visRes'] = spike_counts.cuid.map(rev_visRes_dict)

temp = spikes_df[['cuid', 'region']].drop_duplicates()
spike_counts = pd.merge(spike_counts, temp, on=['cuid'], how='left')

### Save spike count dataframe

In [27]:
spike_counts.to_parquet('operant_spikecounts.parquet')

In [29]:
spike_counts.head()

,cuid,trial,times,spike_count,et,stim,strain,visRes,region
0,CC067431_HP2_185,0.0,0.00,0,CC067431_HP2,0,FX,no,hippo
1,CC067431_HP2_185,0.0,0.01,0,CC067431_HP2,0,FX,no,hippo
2,CC067431_HP2_185,0.0,0.02,0,CC067431_HP2,0,FX,no,hippo
3,CC067431_HP2_185,0.0,0.03,0,CC067431_HP2,0,FX,no,hippo
4,CC067431_HP2_185,0.0,0.04,0,CC067431_HP2,0,FX,no,hippo
